In [1]:
import re
import pandas as pd

print("UK VAT Identifier Discovery")
print("Python environment is ready!")
print("Pandas version:", pd.__version__)

UK VAT Identifier Discovery
Python environment is ready!
Pandas version: 2.2.3


In [2]:
import re

def extract_uk_vat_numbers(text):
    """
    Extract potential UK VAT numbers from text.
    Standard format: GB followed by 9 digits.
    """
    pattern = r'\bGB[\s-]?\d{3}[\s-]?\d{4}[\s-]?\d{2}\b'

    matches = re.findall(pattern, text, flags=re.IGNORECASE)

    vat_numbers = []

    for match in matches:
        clean_number = re.sub(r'[\s-]', '', match).upper()

        if clean_number not in vat_numbers:
            vat_numbers.append(clean_number)

    return vat_numbers


# Test data based on our research cases
sample_text = """
Tesco PLC
VAT Registration Number: GB220430231

Sainsbury's Supermarkets Ltd
VAT Number: GB660454836

Company Registration Number: 00445790
"""

results = extract_uk_vat_numbers(sample_text)

print("Potential UK VAT numbers found:")
for number in results:
    print(number)

Potential UK VAT numbers found:
GB220430231
GB660454836


In [3]:
test_text = """
Company number: 00445790
Phone number: 02079460000
VAT number: GB220430231
Invalid VAT: GB12345678
Another reference: GB1234567890
VAT number: GB660454836
"""

test_results = extract_uk_vat_numbers(test_text)

print("Detected candidates:")
print(test_results)

Detected candidates:
['GB220430231', 'GB660454836']


In [4]:
companies = [
    {
        "company": "Tesco PLC",
        "company_number": "00445790",
        "vat_number": "GB220430231",
        "discovery_source": "Official website - Terms & Conditions",
        "hmrc_status": "Valid"
    },
    {
        "company": "Sainsbury's Supermarkets Ltd",
        "company_number": "03261722",
        "vat_number": "GB660454836",
        "discovery_source": "Official website - Terms & Conditions",
        "hmrc_status": "Valid"
    },
    {
        "company": "BT Group",
        "company_number": None,
        "vat_number": None,
        "discovery_source": "Supplier information",
        "hmrc_status": "Not checked"
    },
    {
        "company": "National Express Limited",
        "company_number": "00232767",
        "vat_number": None,
        "discovery_source": "Terms & Conditions / Contact",
        "hmrc_status": "Not checked"
    }
]

df = pd.DataFrame(companies)

display(df)

,company,company_number,vat_number,discovery_source,hmrc_status
0,Tesco PLC,00445790,GB220430231,Official website - Terms & Conditions,Valid
1,Sainsbury's Supermarkets Ltd,03261722,GB660454836,Official website - Terms & Conditions,Valid
2,BT Group,None,None,Supplier information,Not checked
3,National Express Limited,00232767,None,Terms & Conditions / Contact,Not checked


In [5]:
total_companies = len(df)

discovered = df["vat_number"].notna().sum()

validated = (df["hmrc_status"] == "Valid").sum()

not_discovered = total_companies - discovered

print("RESEARCH SUMMARY")
print("-------------------------")
print("Companies tested:", total_companies)
print("VAT IDs discovered:", discovered)
print("VAT IDs validated by HMRC:", validated)
print("Companies with no VAT ID discovered:", not_discovered)

discovery_rate = (discovered / total_companies) * 100

print(f"Discovery rate: {discovery_rate:.1f}%")

RESEARCH SUMMARY
-------------------------
Companies tested: 4
VAT IDs discovered: 2
VAT IDs validated by HMRC: 2
Companies with no VAT ID discovered: 2
Discovery rate: 50.0%


In [6]:
df.to_csv("uk_vat_research_results.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [7]:
from google.colab import files

files.download("uk_vat_research_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import requests
from bs4 import BeautifulSoup

def extract_page_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; VATResearchBot/1.0)"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        print("HTTP status:", response.status_code)

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for element in soup(["script", "style", "nav", "footer"]):
            element.decompose()

        page_text = soup.get_text(separator=" ", strip=True)

        return page_text

    except requests.RequestException as error:
        print("Page extraction failed:", error)
        return None

In [9]:
tesco_url = "https://www.tesco.com/shop/en-GB/zone/general-terms-and-conditions"

tesco_text = extract_page_text(tesco_url)

if tesco_text:
    print("Extracted characters:", len(tesco_text))
    print("Potential VAT numbers:", extract_uk_vat_numbers(tesco_text))
    print("\nText preview:")
    print(tesco_text[:1000])

HTTP status: 403
Page extraction failed: 403 Client Error: Forbidden for url: https://www.tesco.com/shop/en-GB/zone/general-terms-and-conditions


In [10]:
companies_house_url = "https://find-and-update.company-information.service.gov.uk/company/00445790"

company_text = extract_page_text(companies_house_url)

if company_text:
    print("Extracted characters:", len(company_text))
    print("Potential VAT numbers:", extract_uk_vat_numbers(company_text))
    print("\nText preview:")
    print(company_text[:1000])

HTTP status: 200
Extracted characters: 2210
Potential VAT numbers: []

Text preview:
TESCO PLC overview - Find and update company information - GOV.UK Cookies on Companies House services We use some essential cookies to make our services work. We'd also like to use analytics cookies so we can understand how you use our services and to make improvements. Accept analytics cookies Reject analytics cookies View cookies You've accepted analytics cookies. You can change your cookie settings at any time. Hide this message You've rejected analytics cookies. You can change your cookie settings at any time. Hide this message Cookies on Companies House services We use cookies to make our services work and collect analytics information. To accept or reject analytics cookies, turn on JavaScript in your browser settings and reload this page. Skip to main content GOV.UK Find and update company information Companies House does not check the accuracy of the information filed (link opens a new window) S

In [11]:
sainsburys_url = "https://help.sainsburys.co.uk/help/terms-and-conditions/terms-conditions"

sainsburys_text = extract_page_text(sainsburys_url)

if sainsburys_text:
    print("Extracted characters:", len(sainsburys_text))
    print("Potential VAT numbers:", extract_uk_vat_numbers(sainsburys_text))
    print("\nText preview:")
    print(sainsburys_text[:1000])

HTTP status: 200
Extracted characters: 53803
Potential VAT numbers: []

Text preview:
Terms & Conditions Skip to content Stock Checker Sainsbury’s Home Store Locator Return to Nav Terms & Conditions Terms & Conditions Acceptance of our terms Welcome to Sainsburys.co.uk Terms and Conditions for use. legal notice applies to the contents of the website under the domain name http://www.sainsburys.co.uk/ ("Website"). Our Website is also made up of a number of stores ("Retail Sites") and all orders /purchases made on our Retail Sites will be governed by the specific product terms relevant to each online store or, in their absence, these Terms and Conditions You are permitted to use our Website solely on the basis of these Terms and Conditions. We reserve the right, at our discretion, to update and/or revise these Terms and Conditions. Please check periodically for changes. Important information Product Recalls Important information Product Recalls View more Use of Website Subject to these Te

In [12]:
# Search for VAT-related terms in the extracted page text

vat_keywords = ["VAT", "VAT number", "VAT registration", "Value Added Tax"]

for keyword in vat_keywords:
    count = sainsburys_text.lower().count(keyword.lower())
    print(f"{keyword}: {count} occurrences")

# Display the context around each VAT mention
for match in re.finditer(r"VAT", sainsburys_text, re.IGNORECASE):
    start = max(0, match.start() - 150)
    end = min(len(sainsburys_text), match.end() + 250)
    print("\n--- VAT context ---")
    print(sainsburys_text[start:end])

VAT: 5 occurrences
VAT number: 2 occurrences
VAT registration: 0 occurrences
Value Added Tax: 1 occurrences

--- VAT context ---
Information This website is operated by Sainsbury's Supermarkets Ltd, 33 Charterhouse Street, London, EC1M 6HA, Company Registration Number: 3261722, VAT number: 660 4548 36. Sainsbury's Supermarkets Ltd is registered as a data controller under the terms of the Data Protection Act 2018. Details can be found at www.ico.org.uk Link Opens in New Tab Link Opens in New Tab under number Z4722394. Buying on

--- VAT context ---
burys.co.uk This website is operated by Sainsbury's Supermarkets Ltd, 33 Charterhouse Street, London, EC1M 6HA, Company Registration Number: 3261722, VAT number: 660 4548 36. Sainsbury's Supermarkets Ltd is registered as a data controller under the terms of the Data Protection Act 2018. Details can be found at www.ico.org.uk Link Opens in New Tab Link Opens in New Tab under number Z4722394. Buying on

--- VAT context ---
Prices in 'My basket'

In [13]:
def extract_uk_vat_numbers(text):

    vat_numbers = []

    # Pattern 1: VAT numbers explicitly labelled in the text,
    # with or without the GB prefix.
    labelled_pattern = (
        r'\bVAT(?:\s+(?:registration\s+)?(?:number|no\.?))'
        r'\s*[:#-]?\s*(?:GB\s*)?((?:\d[\s-]*){9})'
    )

    for match in re.finditer(labelled_pattern, text, re.IGNORECASE):
        digits = re.sub(r'\D', '', match.group(1))

        if len(digits) == 9:
            vat_id = "GB" + digits

            if vat_id not in vat_numbers:
                vat_numbers.append(vat_id)

    # Pattern 2: GB-prefixed VAT numbers found elsewhere in the text.
    gb_pattern = r'\bGB[\s-]?\d{3}[\s-]?\d{4}[\s-]?\d{2}\b'

    for match in re.finditer(gb_pattern, text, re.IGNORECASE):
        vat_id = re.sub(r'[\s-]', '', match.group()).upper()

        if vat_id not in vat_numbers:
            vat_numbers.append(vat_id)

    return vat_numbers

In [14]:
sainsburys_results = extract_uk_vat_numbers(sainsburys_text)

print("Sainsbury's VAT discovery results:")
print(sainsburys_results)

Sainsbury's VAT discovery results:
['GB660454836']


In [15]:
print("Extracted VAT candidate:", sainsburys_results[0])
print("Original company: Sainsbury's Supermarkets Ltd")
print("Expected HMRC status: Valid")

Extracted VAT candidate: GB660454836
Original company: Sainsbury's Supermarkets Ltd
Expected HMRC status: Valid


In [16]:
automated_tests = pd.DataFrame([
    {
        "company": "Tesco PLC",
        "source": "Official website",
        "http_status": 403,
        "characters_extracted": 0,
        "vat_discovered": False,
        "hmrc_validation": "Not tested"
    },
    {
        "company": "Tesco PLC",
        "source": "Companies House",
        "http_status": 200,
        "characters_extracted": 2210,
        "vat_discovered": False,
        "hmrc_validation": "Not applicable"
    },
    {
        "company": "Sainsbury's Supermarkets Ltd",
        "source": "Official website",
        "http_status": 200,
        "characters_extracted": len(sainsburys_text),
        "vat_discovered": True,
        "hmrc_validation": "Valid - manually checked"
    }
])

display(automated_tests)

,company,source,http_status,characters_extracted,vat_discovered,hmrc_validation
0,Tesco PLC,Official website,403,0,False,Not tested
1,Tesco PLC,Companies House,200,2210,False,Not applicable
2,Sainsbury's Supermarkets Ltd,Official website,200,53803,True,Valid - manually checked


In [17]:
automated_tests.to_csv(
    "automated_source_test_results.csv",
    index=False
)

print("Automated test report exported successfully!")

Automated test report exported successfully!


In [18]:
from google.colab import files

files.download("automated_source_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>